This function computes the interpolant for the steady/unsteady velocity field. *interpolant_unsteady* is used for unsteady velocity fields, whereas *interpolant_steady* is used for steady velocity fields.

In [ ]:
# import Rectangular bivariate spline from scipy
import numpy as np
from scipy.interpolate import RectBivariateSpline as RBS

class Lagrange5th2D:
    """
    2D tensor-product 5th-order Lagrange interpolator (6x6 stencil),
    compatible with RectBivariateSpline calls ONLY when grid=False.

    Call: obj(yq, xq, grid=False) -> array (N,)
    where yq and xq are paired particle locations.
    """

    def __init__(self, x, y, F):
        self.x = np.asarray(x, dtype=float)   # (Nx,)
        self.y = np.asarray(y, dtype=float)   # (Ny,)
        self.F = np.asarray(F, dtype=float)   # (Ny, Nx)

        if self.x.ndim != 1 or self.y.ndim != 1:
            raise ValueError("x and y must be 1D arrays")
        if self.F.shape != (self.y.size, self.x.size):
            raise ValueError("F must have shape (len(y), len(x))")
        if not (np.all(np.diff(self.x) > 0) and np.all(np.diff(self.y) > 0)):
            raise ValueError("x and y must be strictly increasing")

        self.Nx = self.x.size
        self.Ny = self.y.size

    # -------- MATLAB Basis(p, xp, X, ng) --------
    @staticmethod
    def _basis_6(p_matlab, xp, X):
        """
        p_matlab: integer in MATLAB indexing (2..ng)
        xp: query coordinate (scalar)
        X: 1D grid (numpy)
        Returns:
          L:    (6,) Lagrange weights
          start: python start index of the 6-point stencil in X
        """
        ng = X.size
        i = p_matlab - 1  # python index of p

        # same boundary logic as MATLAB
        if p_matlab == 2:
            start = i - 1          # p-1
        elif p_matlab == ng:
            start = i - 5          # p-5
        elif p_matlab == ng - 1:
            start = i - 4          # p-4
        elif p_matlab == ng - 2:
            start = i - 3          # p-3
        else:
            start = i - 2          # p-2

        # clamp (safety; should already be valid from your logic)
        start = max(0, min(start, ng - 6))

        XCUB = X[start:start + 6]
        if XCUB.size != 6:
            raise RuntimeError(f"Stencil size {XCUB.size}, expected 6. p={p_matlab}, ng={ng}")

        # exact MATLAB numerator/denominator product
        L = np.zeros(6, dtype=float)
        for in_ in range(6):
            num = 1.0
            den = 1.0
            for jn in range(6):
                if jn != in_:
                    num *= (xp - XCUB[jn])
                    den *= (XCUB[in_] - XCUB[jn])
            L[in_] = num / den

        return L, start

    # -------- MATLAB search for cell index j/k --------
    @staticmethod
    def _find_index_like_matlab(xp, X):
        """
        Replicates your MATLAB loops:
        find p in [2..ng] s.t. X(p-1) < xp < X(p) (MATLAB indexing)
        plus overshoot clamp to p=2 or p=ng.
        Returns p in MATLAB indexing.
        """
        ng = X.size
        p = 0

        for m in range(2, ng + 1):
            if (X[m - 2] < xp) and (xp < X[m - 1]):
                p = m
                break

        # overshoot handling (same as MATLAB)
        if X[-1] <= xp:
            p = ng
        if xp <= X[0]:
            p = 2

        # if exactly on edges and still 0 (rare), make it safe
        if p == 0:
            p = 2

        return p

    def __call__(self, yq, xq, grid=False, **kwargs):
        """
        Only supports paired evaluation: grid must be False.
        Returns array of shape (Npoints,).
        """
        if grid is not False:
            raise ValueError("LG5th2D_PairedOnly supports only grid=False (paired particle points).")

        xq = np.atleast_1d(np.asarray(xq, dtype=float))
        yq = np.atleast_1d(np.asarray(yq, dtype=float))

        if xq.shape != yq.shape:
            raise ValueError("With grid=False, xq and yq must have the same shape (paired points).")

        out = np.empty(xq.size, dtype=float)

        for i in range(xq.size):
            xx = xq[i]
            yy = yq[i]

            # find bracketing indices (MATLAB-style)
            j = self._find_index_like_matlab(yy, self.y)  # in y-direction
            k = self._find_index_like_matlab(xx, self.x)  # in x-direction

            # basis weights + stencil starts
            Ly, y_start = self._basis_6(j, yy, self.y)
            Lx, x_start = self._basis_6(k, xx, self.x)

            # extract 6x6 block
            block = self.F[y_start:y_start + 6, x_start:x_start + 6]

            # exact tensor-product sum (same math as your MATLAB double loop)
            out[i] = Ly @ block @ Lx
        
        return out



In [ ]:
from scipy.interpolate import RectBivariateSpline as RBS

def interpolant_unsteady(X, Y, F, method="cubic"):
    """
    Build a list of interpolators for a generic unsteady field F(y,x,t).

    Inputs:
      X, Y: meshgrid arrays (Ny, Nx)
      F:    field array (Ny, Nx, Nt)
    Returns:
      interp_list: list of length Nt, each entry callable like interp(yq, xq, grid=False)
    """
    x = X[0, :]
    y = Y[:, 0]
    Nt = F.shape[2]

    interp_list = []

    if method == "cubic":
        kx = ky = 3
        for j in range(Nt):
            interp_list.append(RBS(y, x, F[:, :, j], kx=kx, ky=ky))

    elif method == "linear":
        kx = ky = 1
        for j in range(Nt):
            interp_list.append(RBS(y, x, F[:, :, j], kx=kx, ky=ky))

    elif method == "LG5th":
        for j in range(Nt):
            interp_list.append(Lagrange5th2D(x, y, F[:, :, j]))  # must support grid=False

    else:
        raise ValueError("method must be 'cubic', 'linear', or 'LG5th'")

    return interp_list


In [ ]:
def interpolant_steady(X, Y, U, V, method = "cubic"):
    '''
    Steady wrapper for scipy.interpolate.RectBivariateSpline. Creates a list of interpolators for u and v velocities
    
    Parameters:
        X: array (Ny, Nx), X-meshgrid
        Y: array (Ny, Nx), Y-meshgrid
        U: array (Ny, Nx), U velocity
        V: array (Ny, Nx), V velocity
        method: Method for interpolation. Default is 'cubic', can be 'linear'
        
    Returns:
        Interpolant: list (2,), U and V  interpolators
    '''
    # Cubic interpolation
    if method == "cubic":
                
        kx = 3
        ky = 3
               
    # linear interpolation
    elif method == "linear":
            
        kx = 1
        ky = 1
            
    # define u, v interpolants
    Interpolant = []
                
    Interpolant.append(RBS(Y[:,0], X[0,:], U, kx=kx, ky=ky))
    Interpolant.append(RBS(Y[:,0], X[0,:], V, kx=kx, ky=ky))  
        
    return Interpolant